# Solrを使うサンプルを動かす前処理

**【注意】途中で手動でのコマンド実行が必要なので、セル1つずつ順に実行をお願いします。**

Solrをベクトルデータベースとして使用するサンプル。

※Solrのバージョン：9.8.1

Solr 9.0以降でベクトル検索機能が追加され、dense_vector型を使用してベクトル検索が可能になりました。

まず、コアを作成し、ベクトル用のフィールドを追加します。

**【注意】途中で手動でのコマンド実行が必要なので、セル1つずつ順に実行をお願いします。**

## 必要パッケージのインポート

In [ ]:
import pysolr
import requests
from sentence_transformers import SentenceTransformer
import time
import json

## 設定

In [ ]:
from solr_conf import *

## Solrコレクションの設定

### コレクションの有無チェック、あれば削除

In [ ]:
# コア一覧取得と削除
try:
    response = requests.get(f'{SOLR_URL}/admin/cores?action=STATUS&wt=json')
    cores_data = response.json()
    cores = list(cores_data.get('status', {}).keys())
    
    if CORE_NAME in cores:
        # コア削除
        delete_response = requests.get(f'{SOLR_URL}/admin/cores?action=UNLOAD&core={CORE_NAME}&deleteIndex=true&deleteDataDir=true&deleteInstanceDir=true&wt=json')
        if delete_response.status_code == 200:
            delete_result = delete_response.json()
            if delete_result.get('responseHeader', {}).get('status') == 0:
                print(f'Core {CORE_NAME} is deleted.')
            else:
                print(f'Delete response: {delete_result}')
        else:
            print(f'Delete failed with status: {delete_response.status_code}')
            
        time.sleep(2)  # 削除完了まで少し待機
    else:
        print(f'Core {CORE_NAME} does not exist.')
except Exception as e:
    print(f'Error checking cores: {e}')

### 【重要】コア作成前にやること！

In [ ]:
print('【超重要！！】ここでコマンド実行が必要です！！\nDockerホストで以下のコマンドを実行し、コアを作成してください。')
print('')
print(f'docker exec llm-rag-examples-solr solr create_core -c {CORE_NAME}')
print(f'docker exec llm-rag-examples-solr solr config -c {CORE_NAME} -s http://localhost:8983 --action set-user-property --property update.autoCreateFields --value false')

In [ ]:
status_response = requests.get(f'{SOLR_URL}/admin/cores?action=STATUS&core={CORE_NAME}&wt=json')
status_result = status_response.json()
status_result

### コレクション作成

In [ ]:
# コア作成（シンプルアプローチ）
print("コア作成を試行します...")

# まず既存のコア状況確認
try:
    status_response = requests.get(f'{SOLR_URL}/admin/cores?action=STATUS&core={CORE_NAME}&wt=json')
    status_result = status_response.json()
    
    # コアが既に存在し、正常に動作している場合
    core_status = status_result.get('status', {}).get(CORE_NAME, {})
    
    if core_status and 'name' in core_status:
        print(f"Core '{CORE_NAME}' already exists and is ready.")
        
        # Schema APIが動作するかテスト
        schema_test = requests.get(f'{SOLR_URL}/{CORE_NAME}/schema?wt=json')
        if schema_test.status_code == 200:
            print(f"✓ Schema API is accessible for core '{CORE_NAME}'.")
        else:
            print(f"⚠ Schema API test failed with status {schema_test.status_code}")
            
    else:
        print(f"Core '{CORE_NAME}' does not exist or is not properly loaded.")
        print("コアを作成するには、Solrコンテナで以下のコマンドを実行してください:")
        print(f"docker exec llm-rag-examples-solr solr create_core -c {CORE_NAME}")
        print("その後、このセルを再実行してください。")
        
except Exception as e:
    print(f"Core status check failed: {e}")

# 作成完了まで少し待機
time.sleep(2)

# 最終確認
try:
    final_check = requests.get(f'{SOLR_URL}/admin/cores?action=STATUS&core={CORE_NAME}&wt=json')
    final_result = final_check.json()
    core_info = final_result.get('status', {}).get(CORE_NAME, {})
    
    if core_info and 'name' in core_info:
        print(f"✓ Core '{CORE_NAME}' is ready for use.")
        print(f"  - Instance Directory: {core_info.get('instanceDir', 'N/A')}")
        print(f"  - Data Directory: {core_info.get('dataDir', 'N/A')}")
        print(f"  - Schema: {core_info.get('schema', 'N/A')}")
    else:
        print(f"✗ Core '{CORE_NAME}' is not ready.")
        raise Exception(f"Core {CORE_NAME} is not available. Please create it manually.")
        
except Exception as e:
    print(f"Final check failed: {e}")
    raise

### スキーマ設定（フィールド追加）

In [ ]:
# まず、ベクトルフィールドタイプを追加
vector_field_type = {
    "add-field-type": {
        "name": "knn_vector",
        "class": "solr.DenseVectorField",
        "vectorDimension": MODEL_DIM,
        "similarityFunction": "cosine",
        "knnAlgorithm": "hnsw"
    }
}

type_response = requests.post(
    f'{SOLR_URL}/{CORE_NAME}/schema',
    json=vector_field_type,
    headers={'Content-Type': 'application/json'}
)

if type_response.status_code == 200:
    try:
        print(f"Vector field type addition response: {type_response.json()}")
    except:
        print(f"Vector field type addition status: {type_response.status_code}")
        print(f"Response text: {type_response.text}")
else:
    print(f"Error adding vector field type: {type_response.status_code}")
    print(f"Response: {type_response.text}")

# テキストフィールドの追加
text_field = {
    "add-field": {
        "name": "text",
        "type": "text_general",
        "stored": True,
        "indexed": True
    }
}

text_response = requests.post(
    f'{SOLR_URL}/{CORE_NAME}/schema',
    json=text_field,
    headers={'Content-Type': 'application/json'}
)

if text_response.status_code == 200:
    try:
        print(f"Text field addition response: {text_response.json()}")
    except:
        print(f"Text field addition status: {text_response.status_code}")
        print(f"Response text: {text_response.text}")
else:
    print(f"Error adding text field: {text_response.status_code}")
    print(f"Response: {text_response.text}")

In [ ]:
# ベクトルフィールドの追加（DenseVectorField型）
vector_field = {
    "add-field": {
        "name": "vector",
        "type": "knn_vector",
        "stored": True,
        "indexed": True
    }
}

vector_response = requests.post(
    f'{SOLR_URL}/{CORE_NAME}/schema',
    json=vector_field,
    headers={'Content-Type': 'application/json'}
)

if vector_response.status_code == 200:
    try:
        print(f"Vector field addition response: {vector_response.json()}")
    except:
        print(f"Vector field addition status: {vector_response.status_code}")
        print(f"Response text: {vector_response.text}")
else:
    print(f"Error adding vector field: {vector_response.status_code}")
    print(f"Response: {vector_response.text}")